In [6]:
from typing import List
import re
import math
import pandas as pd
import os
import json
from transformers import AutoTokenizer, AutoModel

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
input_path = "SGK_Tin12_CD_clean.md"
with open(input_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

In [10]:
import re

output_path = "SGK_Tin12_CD_clean.md"

# Đọc file gốc
with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

clean_text = text

# Xóa các dòng dạng ## Page 42, ### Page 1, # Page 3
clean_text = re.sub(
    r'^\s*#{1,3}\s*Page\s*\d+\s*$', 
    '', 
    clean_text, 
    flags=re.IGNORECASE | re.MULTILINE
)

# Xóa các dòng dạng **Page 35**, *Page 7*, Page 9
clean_text = re.sub(
    r'^\**\s*Page\s*\d+\s*\**$', 
    '', 
    clean_text, 
    flags=re.MULTILINE
)

# Xóa các dòng dạng # --- Trang 7 --- → chuyển thành ---
clean_text = re.sub(
    r'^\s*#\s*-{3}\s*Trang\s*\d+\s*-{3}\s*$',
    '---',
    clean_text,
    flags=re.MULTILINE
)

# Xóa **Trang 12**
clean_text = re.sub(r'^\*\*Trang \d+\*\*\s*$', '', clean_text, flags=re.MULTILINE)

# Xóa # Trang 12
clean_text = re.sub(r'^# Trang \d+\s*$', '', clean_text, flags=re.MULTILINE)

# Xóa ## Trang 7 (pattern mới bổ sung)
clean_text = re.sub(r'^\s*#{2,3}\s*Trang\s*\d+\s*$', '', clean_text, flags=re.MULTILINE)


# Xóa dòng trống thừa sau khi xóa
clean_text = re.sub(r'\n\s*\n+', '\n', clean_text)

# Ghi ra file mới
with open(output_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("✅ Đã loại bỏ tất cả pattern dạng Page và Trang.")


✅ Đã loại bỏ tất cả pattern dạng Page và Trang.


In [ ]:
def _extract_idea_number(self, title: str) -> str:
    """
    Phát hiện số ý/idea từ tiêu đề 
    Ví dụ: "1) ...", "2) ..." -> "1) ...", "2) ..."
    Hoặc: "#### 1. ..." -> "1. ..."
    """
    # Pattern 1: "1) ...", "2) ..." (bắt đầu)
    match = re.match(r'^(\d+\))\s+(.+)', title.strip())
    if match:
        return f"{match.group(1)} {match.group(2)}"
    
    # Pattern 2: "#### 1. ..." (level 4 với số đầu)
    match = re.match(r'^(\d+\.)\s+(.+)', title.strip())
    if match:
        return f"{match.group(1)} {match.group(2)}"
    
    return None


In [12]:
import os
os.chdir('../RawData')  # Chuyển sang thư mục RawData

input_path = "SGK_Tin10_CD_clean.md"
with open(input_path, "r", encoding="utf-8") as f:
    markdown_text = f.read()

os.chdir('../Notebook')  # Quay lại thư mục Notebook

# Khởi tạo chunker với file_name để tự động phát hiện grade
chunker = MarkdownHierarchicalChunker(min_chunk_size=100, merge_short_sections=True, file_name=input_path)
chunks = chunker.chunk_markdown(markdown_text)

print(f"✅ Tổng số chunks: {len(chunks)}\n")

types = Counter([c['metadata']['type'] for c in chunks])
print("📊 Phân loại chunks:")
for chunk_type, count in types.items():
    print(f"  - {chunk_type}: {count}")
print()

# Thống kê metadata
grades = Counter([c['metadata'].get('grade', 'unknown') for c in chunks])
lessons = Counter([c['metadata'].get('lesson') for c in chunks if c['metadata'].get('lesson')])
ideas = Counter([c['metadata'].get('idea') for c in chunks if c['metadata'].get('idea')])

print("\n📚 Phân loại theo Grade:")
for grade, count in sorted(grades.items()):
    print(f"  - Khối {grade}: {count} chunks")

print(f"\n📖 Các bài học: ({len(lessons)} bài)")
for lesson, count in sorted(lessons.items()):
    print(f"  - {lesson}: {count} chunks")

if ideas:
    print(f"\n💡 Các ý chính (idea): ({len(ideas)} ý)")
    for idea, count in sorted(ideas.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  - {idea[:60]}...: {count} chunks" if len(idea) > 60 else f"  - {idea}: {count} chunks")

print("\n" + "="*60)
for i, chunk in enumerate(chunks[:3], 1):
    print(f"\n[CHUNK {i}] Type: {chunk['metadata']['type']}")
    print(f"Grade: {chunk['metadata'].get('grade', 'N/A')} | Lesson: {chunk['metadata'].get('lesson', 'N/A')}")
    print(f"Idea: {chunk['metadata'].get('idea', 'N/A')}")
    print(f"Context: {chunk['context'][:80]}..." if len(chunk['context']) > 80 else f"Context: {chunk['context']}")
    print(f"Level: {chunk['metadata']['level']} | Title: {chunk['metadata']['title']}")
    print(f"Content length: {len(chunk['content'])} chars")
    preview = chunk['content'][:200] + "..." if len(chunk['content']) > 200 else chunk['content']
    print(f"\nContent preview:\n{preview}")
    print("-"*60)

output_path = "rag_chunks.json"
with open(output_path, "a", encoding="utf-8") as f:  # 'w' để ghi đè thay vì append
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã lưu {len(chunks)} chunks vào: {output_path}")


✅ Tổng số chunks: 200

📊 Phân loại chunks:
  - normal: 150
  - merged_with_children: 45
  - short_section: 5


📚 Phân loại theo Grade:
  - Khối 10: 200 chunks

📖 Các bài học: (7 bài)
  - Bài 1: 22 chunks
  - Bài 16: 10 chunks
  - Bài 18: 6 chunks
  - Bài 2: 16 chunks
  - Bài 3: 26 chunks
  - Bài 4: 28 chunks
  - Bài 5: 25 chunks


[CHUNK 1] Type: normal
Grade: 10 | Lesson: Bài 1
Idea: None
Context: # Bài 1: DỮ LIỆU, THÔNG TIN VÀ XỬ LÍ THÔNG TIN
Level: 1 | Title: Bài 1: DỮ LIỆU, THÔNG TIN VÀ XỬ LÍ THÔNG TIN
Content length: 233 chars

Content preview:
**Học xong bài này, em sẽ:**
*   Biết được thông tin là gì, dữ liệu là gì.
*   Phân biệt được thông tin và dữ liệu, nêu được ví dụ minh họa.
*   Biết được xử lí thông tin là gì.
**Em hãy cho biết, thô...
------------------------------------------------------------

[CHUNK 2] Type: normal
Grade: 10 | Lesson: Bài 1
Idea: None
Context: # Bài 1: DỮ LIỆU, THÔNG TIN VÀ XỬ LÍ THÔNG TIN > ## 1) Nguồn thông tin và dữ liệ...
Level: 2 | Title: 1) Nguồ

In [10]:
# Hiển thị chi tiết metadata của một số chunks
print("\n🔍 CHI TIẾT METADATA CỦA MỘT SỐ CHUNKS\n")
print("="*80)

# Chọn các chunks với metadata khác nhau
test_indices = [0, 25, 50, 100, 150, 199]
for idx in test_indices:
    if idx < len(chunks):
        chunk = chunks[idx]
        meta = chunk['metadata']
        print(f"\n[CHUNK {idx}]")
        print(f"  Grade: {meta.get('grade', 'N/A')}")
        print(f"  Lesson: {meta.get('lesson', 'N/A')}")
        print(f"  Idea: {meta.get('idea', 'N/A')}")
        print(f"  Type: {meta.get('type', 'N/A')}")
        print(f"  Title: {meta.get('title', 'N/A')}")
        print(f"  Context: {chunk['context'][:60]}...")
        print(f"  Content length: {len(chunk['content'])} chars")
        print("-"*80)

# Thống kê chunks có idea
chunks_with_idea = [c for c in chunks if c['metadata'].get('idea')]
print(f"\n📊 THỐNG KÊ:\n")
print(f"  - Tổng chunks: {len(chunks)}")
print(f"  - Chunks có idea: {len(chunks_with_idea)}")
print(f"  - Chunks không có idea: {len(chunks) - len(chunks_with_idea)}")

# Hiển thị top ideas
if chunks_with_idea:
    print(f"\n💡 TOP 10 IDEAS:")
    idea_counter = Counter([c['metadata']['idea'] for c in chunks_with_idea])
    for idea, count in idea_counter.most_common(10):
        print(f"  - {idea[:70]}: {count} chunks")



🔍 CHI TIẾT METADATA CỦA MỘT SỐ CHUNKS


[CHUNK 0]
  Grade: 12
  Lesson: Bài 1
  Idea: None
  Type: merged_with_children
  Title: Bài 1: Giới thiệu về Trí tuệ nhân tạo
  Context: # Bài 1: Giới thiệu về Trí tuệ nhân tạo...
  Content length: 7728 chars
--------------------------------------------------------------------------------

[CHUNK 25]
  Grade: 12
  Lesson: None
  Idea: None
  Type: normal
  Title: Chụp cắt lớp
  Context: # Chụp cắt lớp...
  Content length: 308 chars
--------------------------------------------------------------------------------

[CHUNK 50]
  Grade: 12
  Lesson: Bài 4
  Idea: None
  Type: normal
  Title: BÀI 4: TRÌNH BÀY NỘI DUNG THEO DẠNG DANH SÁCH, BẢNG BIỂU
  Context: # BÀI 4: TRÌNH BÀY NỘI DUNG THEO DẠNG DANH SÁCH, BẢNG BIỂU...
  Content length: 155 chars
--------------------------------------------------------------------------------

[CHUNK 100]
  Grade: 12
  Lesson: None
  Idea: None
  Type: normal
  Title: Chắp cánh ước mơ
  Context: # Chắp cánh ước mơ..